# 3. Parallel Document Processing with Ray

This notebook demonstrates parallel document processing using Docling + Ray. This approach is ideal for processing large batches of documents efficiently.

**Key Benefits:**
- Parallel processing across multiple workers
- Automatic load balancing
- Fault tolerance
- Cost-effective scaling on cheap CPU clusters

In [ ]:
# Setup and install dependencies
%pip install docling>=2.68.0 ray>=2.0.0

In [ ]:
# Imports and configuration
import ray
import time
import logging
from pathlib import Path
from typing import List, Dict
from databricks.sdk import WorkspaceClient

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Configuration
CATALOG = "main"
SCHEMA = "default"
RAW_DOCS_VOL = "raw_docs"
PROCESSED_DOCS_VOL = "processed_docs"
MAX_WORKERS = 4
BATCH_SIZE = 5

DOCUMENTS_TABLE = f"{CATALOG}.{SCHEMA}.processed_documents"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.document_chunks"

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"🚀 Ray + Docling Processing Setup")
print(f"Input: /Volumes/{CATALOG}/{SCHEMA}/{RAW_DOCS_VOL}")
print(f"Output: /Volumes/{CATALOG}/{SCHEMA}/{PROCESSED_DOCS_VOL}")
print(f"Workers: {MAX_WORKERS}, Batch size: {BATCH_SIZE}")

In [ ]:
# Ray Actor for parallel document processing
@ray.remote
class DoclingProcessor:
    """Ray actor for parallel document processing with native Docling."""
    
    def __init__(self):
        """Initialize Docling converter."""
        self.converter = DocumentConverter()
        print("Docling processor initialized")
    
    def process_file(self, file_path: str, output_dir: str) -> Dict:
        """Process a single document file."""
        try:
            input_path = Path(file_path)
            output_path = Path(output_dir) / input_path.stem
            output_path.mkdir(parents=True, exist_ok=True)
            
            # Convert with native Docling
            result = self.converter.convert(source=input_path)
            document = result.document
            
            # Save outputs
            document.save_as_json(output_path / "doc.json")
            document.save_as_markdown(output_path / "doc.md")
            
            return {
                "file": str(input_path),
                "status": "success",
                "pages": len(document.pages),
                "pictures": len(document.pictures),
                "tables": len(document.tables),
                "main_text": document.export_to_markdown(),
                "output_path": str(output_path)
            }
            
        except Exception as e:
            return {
                "file": file_path,
                "status": "error", 
                "error": str(e)
            }
    
    def process_batch(self, file_paths: List[str], output_dir: str) -> List[Dict]:
        """Process a batch of files."""
        return [self.process_file(fp, output_dir) for fp in file_paths]

print("✅ DoclingProcessor Ray actor defined")